The problem being addressed is a variation of the Traveling Salesman Problem (TSP), which involves finding the shortest route that visits a given set of locations and returns to the starting point. This problem is being solved using a reinforcement learning approach, in which an agent learns to interact with its environment in order to maximize a reward. In this case, the reward is the minimization of the distance traveled. The agent learns through trial and error, receiving rewards for actions that lead to desirable outcomes and penalties for actions that do not. As the agent gathers more experience, it is able to improve its decision-making and eventually find the optimal solution to the problem.

The environment in which the problem is being solved is large, requiring a significant amount of time to train, with an estimated 300,000 steps(1 episode) taking approximately 24 hours to complete on a Kaggle GPU (P100). However, it is suggested that better results may be achieved by training for 10-20 episodes, though this is uncertain. The provided implementation of the problem is described as being easy to understand and well-commented.

In [ ]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from tqdm import tqdm
import random
from collections import deque
import itertools

from functools import *
from itertools import *

import gc
import threading
import asyncio


pd.options.mode.chained_assignment = None

p = '/kaggle/input/santa-2022/'
sub = pd.read_csv(p+'sample_submission.csv')
df_image = pd.read_csv(p+'image.csv')

In [ ]:
#From https://www.kaggle.com/code/ryanholbrook/getting-started-with-santa-2022

def df_to_image(df):
    side = int(len(df) ** 0.5)  # assumes a square image
    return df_image.set_index(['x', 'y']).to_numpy().reshape(side, side, -1)
image = df_to_image(df_image)

def get_position(config):
    return reduce(lambda p, q: (p[0] + q[0], p[1] + q[1]), config, (0, 0))

def cartesian_to_array(x, y, shape):
    m, n = shape[:2]
    i = (n - 1) // 2 - y
    j = (n - 1) // 2 + x
    if i < 0 or i >= m or j < 0 or j >= n:
        raise ValueError("Coordinates not within given dimensions.")
    return i, j
df = sub[:-1]
df['path'] = df['configuration'].map(lambda x: [list(map(int, p2.split(' '))) for p2 in x.split(';')])
df['point'] = df['path'].map(lambda x: get_position(x))
df['image_point'] = df['path'].map(lambda x: cartesian_to_array(*get_position(x), image.shape))
df['color'] = df['image_point'].map(lambda x : image[x])
df['path'] = df['path'].map(lambda x: np.asarray(x))
print(len(df), 257*257)
df = df.drop_duplicates(subset=['image_point'], keep='first')
print(len(df)) #Some configurations may be helful in reducing load
df.head()

In [ ]:
def compress_path(path):
    r = [[] for _ in range(8)]
    for p in path:
        for i in range(8):
            if len(r[i]) == 0 or r[i][-1] != p[i]:
                r[i].append(p[i])
    mx = max([len(x) for x in r])
    
    for rr in r:
        while len(rr) < mx:
            rr.append(rr[-1])
    r = list(zip(*r))
    for i in range(len(r)):
        r[i] = list(r[i])
    return r
#From https://www.kaggle.com/code/ryanholbrook/getting-started-with-santa-2022
def get_position(config):
    return reduce(lambda p, q: (p[0] + q[0], p[1] + q[1]), config, (0, 0))

def rotate_link(vector, direction):
    x, y = vector
    if direction == 1:  # counter-clockwise
        if y >= x and y > -x:
            x -= 1
        elif y > x and y <= -x:
            y -= 1
        elif y <= x and y < -x:
            x += 1
        else:
            y += 1
    elif direction == -1:  # clockwise
        if y > x and y >= -x:
            x += 1
        elif y >= x and y < -x:
            y += 1
        elif y < x and y <= -x:
            x -= 1
        else:
            y -= 1
    return (x, y)

def rotate(config, i, direction):
    config = config.copy()
    config[i] = rotate_link(config[i], direction)
    return config

def get_square(link_length):
    link = (link_length, 0)
    coords = [link]
    for _ in range(8 * link_length - 1):
        link = rotate_link(link, direction=1)
        coords.append(link)
    return coords

def get_neighbors(config):
    nhbrs = (
        reduce(lambda x, y: rotate(x, *y), enumerate(directions), config)
        for directions in product((-1, 0, 1), repeat=len(config))
    )
    return list(filter(lambda c: c != config, nhbrs))

def reconfiguration_cost(from_config, to_config):
    diffs = np.abs(np.asarray(from_config) - np.asarray(to_config)).sum(axis=1)
    return np.sqrt(diffs.sum())

def color_cost(from_position, to_position, image, color_scale=3.0):
    return np.abs(image[to_position] - image[from_position]).sum() * color_scale

def step_cost(from_config, to_config, image):
    from_position = cartesian_to_array(*get_position(from_config), image.shape)
    to_position = cartesian_to_array(*get_position(to_config), image.shape)
    return (
        reconfiguration_cost(from_config, to_config) +
        color_cost(from_position, to_position, image)
    )

def get_direction(u, v):
    """Returns the sign of the angle from u to v."""
    direction = np.sign(np.cross(u, v))
    if direction == 0 and np.dot(u, v) < 0:
        direction = 1
    return direction

# We don't use this elsewhere, but you might find it useful."""
def get_angle(u, v):
    """Returns the angle (in degreess) from u to v."""
    return np.degrees(np.math.atan2(
        np.cross(u, v),
        np.dot(u, v),
    ))

def get_path_to_point(config, point):
    """Find a path of configurations to `point` starting at `config`."""
    path = [config]
    # Rotate each link, starting with the largest, until the point can
    # be reached by the remaining links. The last link must reach the
    # point itself.
    for i in range(len(config)):
        link = config[i]
        base = get_position(config[:i])
        relbase = (point[0] - base[0], point[1] - base[1])
        position = get_position(config[:i+1])
        relpos = (point[0] - position[0], point[1] - position[1])
        radius = reduce(lambda r, link: r + max(abs(link[0]), abs(link[1])), config[i+1:], 0)
        # Special case when next-to-last link lands on point. 
        if radius == 1 and relpos == (0, 0):
            config = rotate(config, i, 1)
            if get_position(config) == point:
                path.append(config)
                break
            else:
                continue
        while np.max(np.abs(relpos)) > radius:
            direction = get_direction(link, relbase)
            config = rotate(config, i, direction)
            path.append(config)
            link = config[i]
            base = get_position(config[:i])
            relbase = (point[0] - base[0], point[1] - base[1])
            position = get_position(config[:i+1])
            relpos = (point[0] - position[0], point[1] - position[1])
            radius = reduce(lambda r, link: r + max(abs(link[0]), abs(link[1])), config[i+1:], 0)
    assert get_position(path[-1]) == point
                                                                                   
    path = compress_path(path)
    
    return path

def get_path_to_configuration(from_config, to_config):
    path = [from_config]
    config = from_config.copy()
    while config != to_config:
        for i in range(len(config)):
            config = rotate(config, i, get_direction(config[i], to_config[i]))
        path.append(config)
    assert path[-1] == to_config
    return path

# Compute total cost of path over image
def total_cost(path, image):
    return reduce(
        lambda cost, pair: cost + step_cost(pair[0], pair[1], image),
        zip(path[:-1], path[1:]),
        0,
    )

convert_to_float = lambda pos: [pos[0]/128,pos[1]/128]

In [ ]:
def get_1step_neighbors(config):
    # Single-link step:
    neigbors = []
    positions = []
    for i in range(len(config)): #for each arm link
        for d in [-1,1]: #for each direction
            # Rotate link and get new position and vertical displacement:
            config_cur = rotate(config, i, d)
            neigbors.append(config_cur)
            positions.append(get_position(config_cur))
    return neigbors,positions

In [ ]:
init_config = [(64, 0), (-32, 0), (-16, 0), (-8, 0), (-4, 0), (-2, 0), (-1, 0), (-1, 0)]

In [ ]:
class CNNModel(nn.Module):
    def __init__(self, num_classes, hidden_size):
        super(CNNModel, self).__init__()
        
        self.twoD_count = None
        self.oneD_count= None
        
        # layers for image_state
        self.conv1_2D= nn.Conv2d(4, 32, kernel_size=(2,2), padding=(1,1))
        self.conv2_2D = nn.Conv2d(32, 32, kernel_size=(2,2), padding=(1,1))
        self.conv2d_fc1 = nn.Linear(self.foo(self.twoD_count,self.count_neurons2D), 128)  
        self.drop1 = nn.Dropout(0.3)
        self.conv2d_fc2 = nn.Linear(128, 32)   
        
        # layers for config_state  
        self.conv1_1D = nn.Conv1d(2, 16, kernel_size=3)
        self.conv2_1D = nn.Conv1d(16, 32, kernel_size=2)
        self.conv1d_fc1 = nn.Linear(self.foo(self.oneD_count,self.count_neurons1D), 128)
        self.drop2 = nn.Dropout(0.3)
        self.conv1d_fc2 = nn.Linear(128, 32) 
        
        # LSTM layer
        self.lstm = nn.LSTM(32+32, hidden_size)
        
        # combine two inputs
        self.fc = nn.Linear(hidden_size, 32)  
        self.drop4 = nn.Dropout(0.5)
        self.fc2 = nn.Linear(32, num_classes)     
        
    def count_neurons2D(self):
        x = torch.rand(1,4,257,257)
        x = self.conv1_2D(x)
        x = self.conv2_2D(x)
        out = torch.prod(torch.tensor(x.shape))
        del x
        torch.cuda.empty_cache()
        return out
    def count_neurons1D(self):
        x = torch.rand(1,2,9)
        x = self.conv1_1D(x)
        x = self.conv2_1D(x)
        out = torch.prod(torch.tensor(x.shape))
        del x
        torch.cuda.empty_cache()
        return out
    
    def foo(self,count_var,function):
        if count_var is None:
            count = function()
            count_var = count
        return count_var
            
    
    
    def forward(self, image_states,config_states):
#         print(image_states.shape,config_states.shape)
        x1 = self.conv1_2D(image_states)
        x1 = self.conv2_2D(x1)
        x1 = x1.view(x1.size(0), -1)
        x1 = self.conv2d_fc1(x1)
        drop1 = self.drop1(x1)
        x1 = self.conv2d_fc2(drop1)
        
        x2 = self.conv1_1D(config_states)
        x2 = self.conv2_1D(x2)
        x2 = x2.view(x2.size(0), -1)
        x2 = self.conv1d_fc1(x2)
        drop2 = self.drop2(x2)
        x2 = self.conv1d_fc2(drop2)
        
        # combine two inputs and pass through LSTM layer
        x = torch.cat((x1, x2), dim=1)
#         print(x.shape)
        x, (hn, cn) = self.lstm(x)
        
        # take the last hidden state of the LSTM layer
        x = hn[-1]
#         print(x.shape)
        x = self.fc(x)
        drop4 = self.drop4(x)
        x = self.fc2(drop4)
        return x



In [ ]:
class Agent():
    def __init__(self ,gamma ,epsilon ,batch_size ,n_actions,online_net,
                 target_net,max_mem_size = 2000,min_mem_size=1500 ,eps_end = 0.01 ,
                 eps_dec= 5e-4,target_net_update_freq=500,double=True):
        
        # discount factor for future rewards
        self.gamma = gamma
        # exploration rate
        self.epsilon = epsilon
        # minimum exploration rate
        self.eps_min = eps_end
        # exploration rate decay rate
        self.eps_dec = eps_dec
        # list of all possible actions
        self.action_space = [i for i in range(n_actions)]
        # maximum size of memory buffer
        self.max_mem_size = max_mem_size
        # minimum size of memory buffer before learning starts
        self.min_mem_size = min_mem_size
        # batch size for learning
        self.batch_size = batch_size
        # counter for storing transitions in memory
        self.mem_cntr = 0
        # online network for approximating Q-function
        self.online_net = online_net
        # target network for stabilizing learning
        self.target_net = target_net
        # memory buffer for storing transitions
        self.memory = deque(maxlen=self.max_mem_size)
         # frequency for updating the target network
        self.target_net_update_freq = target_net_update_freq
        self.optimizer = optim.Adam(self.online_net.parameters(), lr=0.0003)
        self.loss_function = nn.MSELoss()
        # device to use for tensor computations (GPU if available, else CPU)
        self.device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
        # flag for using Double DQN variation
        self.double = double
    
    def store_transitions(self,image_state,config_state ,action,reward ,done ,new_image_state,new_config_state):
        transitions = (image_state,config_state ,action,reward ,done ,new_image_state,new_config_state)
        # store a transition in the memory buffer
        self.memory.append(transitions)
        self.mem_cntr+=1
        
    def choose_action(self,n_actions,image_state,config_state):
        self.n_actions = n_actions

        if self.epsilon < np.random.random():
            # choose action with highest expected reward (exploitation)
            # convert image state and config state to tensors and send to device
            image_state = image_state.to(dtype = torch.float32,device = device)
            config_state = config_state.to(dtype = torch.float32,device = device)
            # get Q-values for all actions from online network
            q_values = self.online_net(image_state,config_state)
            # choose action with highest Q-value
            action = torch.argmax(q_values).item()
        else:
            # choose random action (exploration)
            action = np.random.choice([i for i in range(self.n_actions)])
        
        return action
    
    async def learn(self,step):
        # check if memory buffer is large enough to start learning
        if self.mem_cntr < self.min_mem_size:
            return
        
        start = random.randint(0,len(agent.memory) - agent.batch_size) 
        transitions = list(agent.memory)[start:start+agent.batch_size] # took a sequnce
        
        # create tensors for image states, config states, actions, rewards, and dones
        image_states_tensor = torch.cat(tuple([t[0] for t in transitions]), dim=0).to(device = agent.device,dtype = torch.float32)
        config_states_tensor = torch.cat(tuple([t[1] for t in transitions]), dim=0).to(device = agent.device,dtype = torch.float32)
        actions = transitions[-1][2]
        rewards = transitions[-1][3]
        dones = transitions[-1][4]
        new_image_states_tensor = torch.cat(tuple([t[5] for t in transitions]), dim=0).to(device = agent.device,dtype = torch.float32)
        new_config_states_tensor = torch.cat(tuple([t[6] for t in transitions]), dim=0).to(device = agent.device,dtype = torch.float32)

        actions_tensor = torch.tensor(actions ,dtype = torch.int64).unsqueeze(-1)
        rewards_tensor = torch.tensor(rewards ,dtype = torch.float32).unsqueeze(-1)
        dones_tensor = torch.tensor(dones,dtype = torch.float32).unsqueeze(-1)

        if agent.double:
            q_eval_target = agent.online_net(new_image_states_tensor,new_config_states_tensor).to("cpu")
            q_next_target = agent.target_net(new_image_states_tensor,new_config_states_tensor).to("cpu")
            next_actions_tensor = torch.argmax(q_eval_target,dim=0).unsqueeze(-1).data.cpu()
            action_q_values = torch.gather(input = q_next_target ,dim=0 ,index = actions_tensor)
            q_target = rewards_tensor + agent.gamma * action_q_values * (1-dones_tensor)
        else:
            q_next = self.target_net(new_image_states_tensor,new_config_states_tensor).to("cpu")
            max_q_next = q_next.max(dim=1 ,keepdim=True)[0]
            q_target = rewards_tensor + agent.gamma * max_q_next * (1-dones_tensor)
        q_eval = agent.online_net(image_states_tensor,config_states_tensor).to("cpu")
        action_q_values = torch.gather(input = q_eval ,dim=0 ,index = actions_tensor)
        del image_states_tensor,config_states_tensor,new_image_states_tensor,new_config_states_tensor
        torch.cuda.empty_cache()
        agent.optimizer.zero_grad()
        loss= agent.loss_function(action_q_values.to(agent.device) ,q_target.to(agent.device)).to(agent.device) 
        loss.backward()
        agent.optimizer.step()
        if step % self.target_net_update_freq == 0:
            self.target_net.load_state_dict(self.online_net.state_dict())
        self.epsilon = self.epsilon - self.eps_dec if self.epsilon > self.eps_min else self.eps_min
    
    async def start_learning(self, step):
        await self.learn(step)

In [ ]:
def update_img(img,pos):
    # set the last channel (alpha channel) of the pixel at position 'pos' to 1
    # to indicate it's filled
    img[pos[0]][pos[1]][-1] = 1
    # transpose the image array to put the channels in the first dimension.
    img = np.transpose(img,(2,0,1))
    # convert the image array to a PyTorch tensor
    img = torch.tensor(img)
    return img

def create_conf_state(config,pos):
    # normalize the config values by dividing by 64
    config_arr = np.array(config)/64
    # normalize the position values by dividing by 256
    pos_arr = np.array(pos)/256
    # insert the position values at the end of the config array
    config_arr = np.insert(config_arr,config_arr.shape[0],pos_arr,axis=0)
    # transpose the config array to have the config values in the first dimension
    config_arr = np.transpose(config_arr,(1,0))
    config_ten = torch.tensor(config_arr)
    return config_ten

In [ ]:
class Env():
    def __init__(self):
        # initialize the starting configuration of the robot
        self.init_config = [(64, 0), (-32, 0), (-16, 0), (-8, 0), (-4, 0), (-2, 0), (-1, 0), (-1, 0)]
        self.image = image
        # initialize an empty list to store filled coordinates
        self.filled_coordinates = []
        # initialize the step counter to 0
        self.step_count = 0
        
    def reset(self):
        # reset the step counter
        self.step_count = 0
        # create the initial image state by adding an alpha channel to the image array
        initial_img = np.insert(self.image, self.image.shape[2], 0, axis=2)
        # update the initial image state with the robot's starting position
        img_state = update_img(initial_img,(0,0))
        # create the initial config state
        config_state = create_conf_state(self.init_config,(0,0))
        
        # return the initial image and config states as a tuple
        reset = img_state,config_state
        return reset
        
        
    def step(self,action,current_config,image_state):
        # get the possible configurations and positions after taking one step from the current configuration
        neighbors,positions = get_1step_neighbors(current_config)
        new_config = neighbors[action]
        new_pos = get_position(new_config)
        pos = cartesian_to_array(new_pos[0],new_pos[1],image.shape)
        # transpose the image state array to put the channels in the last dimension
        image_state = np.transpose(image_state,(1,2,0))
        # update the image state with the new position
        new_image_state = update_img(image_state,pos)
        # create the new config state
        new_config_state = create_conf_state(new_config,pos)
        
        # initialize the reward to 5
        reward = 5
        
        from_position = cartesian_to_array(*get_position(current_config), image.shape)
        to_position = cartesian_to_array(*get_position(new_config), image.shape)
        # subtract the color cost between the positions from the reward
        reward -= color_cost(from_position, to_position, image)
        # subtract a small value for each step taken. (Time Penalty)
        reward -= 0.001 * self.step_count 
        """
        In here i don't use reconfiguration cost, because reconfiguration cost is same for each step, because we only taking one step
        """
        
        if new_pos in self.filled_coordinates:
            # subtract a large value from the reward if the position is already filled
            reward -= 4.5
        
        else:
            self.filled_coordinates.append(new_pos)
            
        if len(set(self.filled_coordinates)) == 66049:
            # If the number of unique filled coordinates is equal to 66049, the reward is increased by 100.
            reward += 100
        done = False
        
        
        # If the number of unique filled coordinates is greater than or equal to 66049 or the step count is greater than 250000, the done flag is set to True
        if (len(set(self.filled_coordinates)) >= 66049 or self.step_count > 10): # please convert 10 to 300000
            done = True            
            
        info = {
            "filled_coordinates": len(set(self.filled_coordinates)),
            "done": done
        }
        
        return (new_image_state,new_config_state,new_config,reward,done,info,self.step_count)

In [ ]:
# del online_net
# del target_net
# del agent
# del env
torch.cuda.empty_cache()
gc.collect()


In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
n_actions = 16
online_net = CNNModel(n_actions,256).to(device)
target_net = CNNModel(n_actions,256).to(device)
target_net.load_state_dict(online_net.state_dict())

In [ ]:
agent = Agent(gamma=0.98 ,epsilon=1.0 ,batch_size=3
              ,n_actions=n_actions ,eps_end=0.01,
              online_net=online_net,target_net=target_net,double=True)
          
agent.memory.clear()

In [ ]:
episodes = 1
reward_buffer = deque([0.0] ,maxlen=100)
n_actions  =16

for episode in range(episodes):
    env = Env()
    image_state,config_state = env.reset()
    image_state = image_state.view(-1,4,257,257)
    config_state = config_state.view(-1,2,9)
    image_states = torch.cat([image_state for _ in range(3)], dim=0).reshape(3,4,257,257)
    config_states = torch.cat([config_state for _ in range(3)], dim=0).reshape(3,2,9)
    config = env.init_config
    score = 0
    step = 0
    prev_score = 0
    prev_filled_points = 0
    cost = 0
    while True:
        action = agent.choose_action(n_actions,image_states,config_states)
        new_image_state,new_config_state,new_config,reward,done,info,step_count = env.step(action,
                                                                                     config,
                                                                                    image_state[0]
                                                                                          )
        new_image_state = new_image_state.view(-1,4,257,257) 
        new_config_state = new_config_state.view(-1,2,9)
        new_image_states = torch.cat((image_states[1:,:,:,:] ,new_image_state.view(-1,4,257,257)))[:,:,:,:].to(torch.float32)
        new_config_states = torch.cat((config_states[1:,:,:] ,new_config_state.view(-1,2,9)))[:,:,:].to(torch.float32)
        agent.store_transitions(image_state = image_state,
                                config_state =  config_state,
                                action = action,
                                reward = reward ,
                                done = done,
                                new_image_state = new_image_state,
                                new_config_state = new_config_state
                               )
        
        cost += step_cost(config,new_config,image)
        image_state = new_image_state
        config_state = new_config_state
        config_states = new_config_states
        image_states = new_image_states
        config = new_config
        score += reward
        env.step_count += 1
        await agent.start_learning(step)
        
        if done:
            reward_buffer.append(score)
            break
            
            

        if env.step_count % 1000 == 0:
            print()
    #         print(f"step: {episode}   avarage_reward: {np.mean(reward_buffer)}  epsilon: {agent.epsilon} ")
            print(f"step: {env.step_count}   reward: {score + prev_score}  epsilon: {agent.epsilon} Filled Points: {len(env.filled_coordinates)-prev_filled_points}")
            print(f"Total Filled Points: {len(env.filled_coordinates)}")
            prev_score = -score
            prev_filled_points = len(env.filled_coordinates)
        
        if env.step_count % 100000 == 0:
            torch.save(online_net, 'online_net.pt')
            torch.save(target_net, 'target_net.pt')
        
    print("#" * 100)        
    print(score)
    print(cost)
    torch.save(online_net, 'online_net.pt')
    torch.save(target_net, 'target_net.pt')
    
print("# please convert 10 to 300000 in Env.step. if not it will only take 10 step and finish the episode")

In [ ]:
import torch

# Save the model
torch.save(online_net, 'online_net.pt')
torch.save(target_net, 'target_net.pt')

# # Load the model
# online_net = torch.load('online_net.pt')
# target_net = torch.load('target_net.pt')